In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/sales_forecast.csv", encoding="latin1")

In [4]:
print(df.columns)

Index(['Date', 'Forecast Sales'], dtype='str')


In [5]:
df = df.rename(columns={
    "Forecast Sales": "forecast"
})

In [6]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

In [7]:
#Rolling statistics
df["rolling_mean"] = df["forecast"].rolling(window=7).mean()
df["rolling_std"] = df["forecast"].rolling(window=7).std()

In [9]:
#Detecting anomalies
import numpy as np
df["anomaly"] = np.abs(df["forecast"] - df["rolling_mean"]) > (2 * df["rolling_std"])

In [10]:
#Handling NaN rows
df = df.dropna()

In [11]:
#Classifying anomalies
df["anomaly_type"] = "normal"

df.loc[df["anomaly"], "anomaly_type"] = "forecast_anomaly"

In [12]:
#adding a explanation column
def explain(row):
    if row["anomaly"]:
        return "Forecast deviates significantly from recent trend"
    return "Normal behavior"

df["anomaly_reason"] = df.apply(explain, axis=1)

In [13]:
#anomaly table
anomalies = df[df["anomaly"]].copy()

In [15]:
#saving output
df.to_csv("../data/processed/anomalies_full.csv", index=False)
anomalies.to_csv("../data/processed/anomaly_events.csv", index=False)